# DEPENDENCIES & FABRIC CONFIGURATION

In [ ]:
from fabrictestbed_extensions.fablib.fablib import fablib, FablibManager as fablib_manager
import time, os

file_path = 'project_id'
try:
    with open(file_path, 'r', 
              encoding='utf-8') as f:
        content = f.read()
        fablib = fablib_manager(project_id=content)
        slice_name = 'dist_trader'
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")

In [ ]:
try:
    fablib.delete_slice(slice_name)
    time.sleep(10)
except:
    pass

# NODE CONFIGURATION

## Slice Initialization

In [ ]:
slice = fablib.new_slice(name=slice_name)
site = 'SEAT'

node_configs = [
    ('ingestion-node', 'ingestion.py'),  # also hosts Kafka/Redis
    ('strategy-node',  'strategy.py'),
    ('risk-node',      'risk.py'),
    ('execution-node', 'execution.py'),
    ('monitor-node',   'dashboard.py'),
    ('monolithic-node', 'monolithic.py')
]

nodes = {}
for name, _ in node_configs:
    nodes[name] = slice.add_node(name=name, site=site, image='default_ubuntu_22')
    nodes[name].set_capacities(cores=2, ram=4, disk=10)

net = slice.add_l2network(name="trading_lan")
for i, (name, node) in enumerate(nodes.items()):
    iface = node.add_component(model='NIC_Basic', name=f'{name}_nic').get_interfaces()[0]
    net.add_interface(iface)

slice.submit(wait=True, wait_timeout=600, wait_interval=10, progress=True)

## Node Dependencies

In [ ]:
slice_name = "dist_trader"
slice = fablib.get_slice(name=slice_name)
nodes = slice.get_nodes()

# sort nodes to ensure 'ingestion-node' is always index 0 (10.0.0.1)
nodes.sort(key=lambda x: x.get_name())

print("Configuring Network and Installing Dependencies...")

for i, node in enumerate(nodes):
    print(f"Working on {node.get_name()}...")

    node.execute("sudo apt update -y -qq")
    node.execute("sudo apt install -y build-essential net-tools python3-pip -qq")
    node.execute("pip3 install confluent-kafka pydantic redis -qq")

ingestion = slice.get_node(name="ingestion-node")
print("Installing Kafka/Redis on Ingestion Node...")
ingestion.execute("sudo apt-get install -y redis-server default-jre")
ingestion.execute("wget https://archive.apache.org/dist/kafka/3.6.0/kafka_2.13-3.6.0.tgz")
ingestion.execute("tar -xzf kafka_2.13-3.6.0.tgz")

scripts = {
    "ingestion-node": "ingestion.py",
    "strategy-node": "strategy.py",
    "risk-node": "risk.py",
    "execution-node": "execution.py",
    "monitor-node": "dashboard.py",
    "monolithic-node": "monolithic.py"
}
for node in nodes:
    name = node.get_name()
    if name in scripts:
        node.execute('rm -f *.py *.log')
        node.upload_file('models.py', 'models.py')
        node.upload_file(scripts[name], scripts[name])

# DEPLOYMENT

## Socket and IP Tweaks

In [ ]:
for name, _ in node_configs:
    node = slice.get_node(name=name)
    print(f"Configuring IP for {name}...")
    
    iface = node.get_interface(network_name="trading_lan")
    os_iface = iface.get_os_interface()
    
    # assign the internal IP based on the nodes role and order
    ip_last_octet = {"ingestion-node": 1, "strategy-node": 2, "risk-node": 3, "execution-node": 4, "monitor-node": 5, "monolithic-node": 6}
    target_ip = f"10.0.0.{ip_last_octet.get(name, 10)}"
    
    config_cmds = [
        f"sudo ip addr flush dev {os_iface}",
        f"sudo ip addr add {target_ip}/24 dev {os_iface}",
        f"sudo ip link set dev {os_iface} up"
    ]
    
    for cmd in config_cmds:
        node.execute(cmd)
    
    print(f"  - {name} is now at {target_ip} on {os_iface}")

## Environment Wipe & Script Execution

In [ ]:
slice_name = "dist_trader"
slice = fablib.get_slice(name=slice_name)
nodes = slice.get_nodes()

node_configs = [
    ('ingestion-node', 'ingestion.py'),  # also hosts Kafka/Redis
    ('strategy-node',  'strategy.py'),
    ('risk-node',      'risk.py'),
    ('execution-node', 'execution.py'),
    ('monitor-node',   'dashboard.py'),
    ('monolithic-node', 'monolithic.py')
]

print("--- Force Killing All Processes ---")
for name, _ in node_configs:
    node = slice.get_node(name=name)
    node.execute("sudo pkill -9 -f python3 || true")
    node.execute("sudo pkill -9 -f kafka || true")
    node.execute("sudo pkill -9 -f zookeeper || true")
    node.execute("sudo pkill -9 -f redis-server || true")

print("--- Wiping Memory Logs ---")
ingestion = slice.get_node(name='ingestion-node')
ingestion.execute("sudo rm -rf /tmp/kafka-logs /tmp/zookeeper")
ingestion.execute("sudo rm -f /var/run/redis/redis-server.pid || true")

config = """
broker.id=0
zookeeper.connect=localhost:2181
listeners=PLAINTEXT://10.0.0.1:9092
advertised.listeners=PLAINTEXT://10.0.0.1:9092
log.dirs=/tmp/kafka-logs
zookeeper.connection.timeout.ms=18000
offsets.topic.replication.factor=1
transaction.state.log.replication.factor=1
transaction.state.log.min.isr=1
"""
ingestion.execute(f"echo '{config}' > ~/kafka_2.13-3.6.0/config/server.properties")

print("--- Starting Message Bus & Redis ---")
ingestion.execute("nohup ~/kafka_2.13-3.6.0/bin/zookeeper-server-start.sh ~/kafka_2.13-3.6.0/config/zookeeper.properties > zk.log 2>&1 &")
time.sleep(5)
ingestion.execute("nohup ~/kafka_2.13-3.6.0/bin/kafka-server-start.sh ~/kafka_2.13-3.6.0/config/server.properties > kafka.log 2>&1 &")

print("  > Restarting Redis...")
ingestion.execute("sudo systemctl restart redis-server &") 
time.sleep(3)
ingestion.execute("redis-cli flushall")

print("--- Launching Node Scripts ---")
for name, script in node_configs:
    node = slice.get_node(name=name)
    node.execute('rm -f *.py *.log')
    node.upload_file('models.py', 'models.py')
    node.upload_file(script, script)
    print(f"  > Launching {script} on {name}...")
    node.execute(f"nohup python3 -u {script} > {name}.log 2>&1 & disown")

print("\nSystem Cleaned and Redeployed.")

# VALIDATION

Message Bus Peeking

In [ ]:
stages = [
    ("PRICES", "prices.TICK_0"),
    ("SIGNALS", "signals"),
    ("ORDERS", "orders"),
    ("FILLS", "fills"),
    ("MONOLITHIC STATE", "monolithic_stats")
]

for name, topic in stages:
    print(f"--- Checking {name} ({topic}) ---")
    out = ingestion.execute(f"~/kafka_2.13-3.6.0/bin/kafka-console-consumer.sh --topic {topic} --bootstrap-server 10.0.0.1:9092 --from-beginning --max-messages 1 --timeout-ms 10000")
    print(out if out else "EMPTY\n")

Log Peeking

In [ ]:
for name, _ in node_configs:
    node = slice.get_node(name=name)
    print(f'---------- {name} ----------')
    node.execute(f'tail -n 10 {name}.log')

Monitor SSH

In [ ]:
monitor = slice.get_node(name='monitor-node')

argv = ['/home/fabric/work/fabric_config/slice_key',
       '/home/fabric/work/fabric_config/ssh_config']
user = monitor.get_username()
ip = str(monitor.get_management_ip())

os.system('chmod 600 /home/fabric/work/fabric_config/slice_key')
os.system('chmod 600 /home/fabric/work/fabric_config/fabric_bastion_key')
print(f'ssh -i {argv[0]} -F {argv[1]} {user}@{ip}')